#Chapter 7.4:String Manipulation  
##You are working as a Machine Learning Engineer for the Eclipse community platform. Your automated bot has recorded raw user activity logs from the #feedback-and-support channel.

The dataset is extremely messy. It contains raw text strings with user tags, system error placeholders, email addresses embedded inside sentences, unnecessary white spaces, and spam messages containing malicious links.

Your task is to build a clean Data Preparation Pipeline using Pandas string vectorization (.str accessor methods, Regex extraction, splitting, and filtering) to clean the dataset before passing it into an NLP (Natural Language Processing) sentiment model.

In [4]:
import pandas as pd
import numpy as np

# Synthetic Production Discord Logs Dump 
raw_discord_data = {
    'log_id': ['LOG_101', 'LOG_102', 'LOG_103', 'LOG_104', 'LOG_105', 'LOG_106'],
    'user_info': ['  Zaraki_Dev#1234  ', 'ECLIPSE_USER#9999 ', '  shadow_ninja#0001', 'bot_attacker#1111', '  Alex_Support#5555  ', np.nan],
    'raw_message': [
        ' [URGENT] I cannot access the Python course! Send details to zaraki.dev@eclipse.ro please! ',
        '[SPAM] Make money fast at http://crypto-scam.com !!! email contact: scam@fake.com',
        '   [FEEDBACK] Great platform, but I have a payment issue. mail: shadow@yahoo.com ',
        'BUY CHEAP BITCOIN http://scam-site.org FAST PROMO!!!',
        '[URGENT] I got a pandas code error [ERROR_404]. Contact: alex.sup@gmail.com  ',
        np.nan
    ]
}

df_discord = pd.DataFrame(raw_discord_data)

print("=== RAW DISCORD LOGS ===")
print(df_discord)
print("=" * 60)

#Task 1:User Identity Parsing & Normalization
df_discord['user_info'] = df_discord['user_info'].str.strip()
df_discord[['username','discriminator']]= df_discord['user_info'].str.split('#',expand=True)
df_discord['username'] = df_discord['username'].str.lower()

#Task 2: Category Label Extraction
df_discord['category'] = df_discord['raw_message'].str.extract(r'\[([A-Z]+)\]',expand=True)

#Task 3: Spam Detection & Row Removal
df_discord = df_discord[~df_discord['raw_message'].str.contains('http','PROMO',na=False,regex=False)]

#Task 4: Email Address Harvesting
df_discord['user_email'] = df_discord['raw_message'].str.extract(r'(\S+@\S+\.\S+)',expand=True)

#Task 5: Message Text Sanitization
df_discord['raw_message'] = df_discord['raw_message'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
df_discord

=== RAW DISCORD LOGS ===
    log_id              user_info  \
0  LOG_101      Zaraki_Dev#1234     
1  LOG_102     ECLIPSE_USER#9999    
2  LOG_103      shadow_ninja#0001   
3  LOG_104      bot_attacker#1111   
4  LOG_105    Alex_Support#5555     
5  LOG_106                    NaN   

                                         raw_message  
0   [URGENT] I cannot access the Python course! S...  
1  [SPAM] Make money fast at http://crypto-scam.c...  
2     [FEEDBACK] Great platform, but I have a pay...  
3  BUY CHEAP BITCOIN http://scam-site.org FAST PR...  
4  [URGENT] I got a pandas code error [ERROR_404]...  
5                                                NaN  


,log_id,user_info,raw_message,username,discriminator,category,user_email
0,LOG_101,Zaraki_Dev#1234,I cannot access the Python course! Send detail...,zaraki_dev,1234,URGENT,zaraki.dev@eclipse.ro
2,LOG_103,shadow_ninja#0001,"Great platform, but I have a payment issue. ma...",shadow_ninja,0001,FEEDBACK,shadow@yahoo.com
4,LOG_105,Alex_Support#5555,I got a pandas code error . Contact: alex.sup@...,alex_support,5555,URGENT,alex.sup@gmail.com
5,LOG_106,NaN,NaN,NaN,NaN,NaN,NaN
